# 1. Introducción: Análisis y Procesamiento Integral del Conflicto

Este notebook profesional tiene un propósito doble:
1. **Exploratory Data Analysis (EDA):** Analizar en profundidad el dataset geopolítico (país-día) de la escalada del conflicto entre **Irán, Israel y EE.UU.** mediante visualizaciones avanzadas robustas frente a valores extremos (outliers).
2. **Data Preprocessing Pipeline:** Aplicar técnicas de ingeniería de características, transformaciones matemáticas, codificación y escalamiento para generar un dataset completamente depurado y **listo para el modelado de Machine Learning**.

### Objetivo Predictivo
Predecir el nivel de escalada (0 = Baja, 1 = Media, 2 = Alta) combinando fuentes heterogéneas:
*   Eventos físicos conflictivos (ACLED)
*   Tono y negatividad mediática internacional (GDELT)
*   Dinámica macroeconómica (Precio del Brent)
*   Señales semánticas tempranas extraídas mediante NLP (Embeddings lingüísticos)


# 2. Carga de Librerías y Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# Configuración visual para reportes
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (12, 6), 
    'axes.titlesize': 14, 
    'axes.labelsize': 12,
    'axes.titleweight': 'bold',
    'legend.frameon': True
})
# Semáforo para Escalada: Verde (Baja), Amarillo (Media), Rojo oscuro (Alta)
escalada_colors = ['#2ECC71', '#F1C40F', '#C0392B']


In [ ]:
# Cargar el dataset
df = pd.read_csv('dataset_final_con_embeddings.csv')

# Conversión de fechas
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'])

# Ordenar cronológicamente para evitar problemas en series de tiempo
df = df.sort_values(by=['fecha', 'pais']).reset_index(drop=True)

print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas")
display(df.head())


# 3. Calidad de Datos (Completeness)

Validamos la integridad estructural de la información. Reemplazamos el tradicional "heatmap" por un gráfico de barras de completitud que es mucho más interpretable para datasets casi perfectos.


In [ ]:
# Cálculo de Valores Nulos
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

if null_percentages.sum() == 0:
    print("El dataset es de excelente calidad: No hay valores nulos presentes en las variables.")
else:
    # Gráfico de completitud de datos (Porcentaje de datos NO nulos)
    completitud = 100 - null_percentages
    completitud = completitud.sort_values()
    
    plt.figure(figsize=(10, 8))
    ax = sns.barplot(x=completitud.values, y=completitud.index, palette='viridis')
    plt.axvline(x=100, color='red', linestyle='--', linewidth=2)
    plt.title('Completitud de Datos por Variable (%)')
    plt.xlabel('Porcentaje de Datos Válidos')
    plt.ylabel('Variable')
    plt.xlim(0, 105)
    plt.show()


# 4. Análisis Univariado: Manejo Visual de Extremos

Los conflictos bélicos se rigen por la **Ley de Potencias (Power Laws)**: casi todos los días ocurren cero o muy pocos eventos, y repentinamente surgen "Cisnes Negros" (días con cientos de bajas). 
Para interpretar estas distribuciones sin que la gráfica colapse visualmente, aplicamos escalas **Logarítmicas (log1p)** y usamos **Boxenplots** en lugar de Boxplots tradicionales.


In [ ]:
num_vars_extremos = [col for col in ['n_eventos', 'total_bajas', 'precio_brent', 'tono_gdelt'] if col in df.columns]

if num_vars_extremos:
    fig, axes = plt.subplots(len(num_vars_extremos), 2, figsize=(15, 4 * len(num_vars_extremos)))
    if len(num_vars_extremos) == 1: axes = np.array([axes])

    for i, var in enumerate(num_vars_extremos):
        # Es asimétrica? (variables de recuento de muertes o eventos)
        is_skewed = df[var].skew() > 3
        
        # 1. Histograma (Si es asimétrica, usamos escala log en el eje Y)
        ax_hist = sns.histplot(df[var], kde=False, bins=40, ax=axes[i, 0], color='steelblue')
        if is_skewed:
            ax_hist.set_yscale('symlog')
            ax_hist.set_ylabel('Frecuencia (Escala SymLog)')
            ax_hist.set_title(f'Distribución (Asimétrica) de {var}')
        else:
            ax_hist.set_title(f'Distribución de {var}')
            
        # 2. Boxenplot (Excelente para visualizar muchos outliers sin perder la mediana real)
        sns.boxenplot(x=df[var], ax=axes[i, 1], color='coral')
        axes[i, 1].set_title(f'Boxenplot de {var} (Robusto a Outliers)')

    plt.tight_layout()
    plt.show()


**Insights:** Las variables de conteo muestran un fuerte sesgo a la derecha. Esto nos alerta que en la fase de preprocesamiento **deberemos aplicar transformaciones logarítmicas matemáticas** antes de alimentar los modelos de ML, de lo contrario, los modelos lineales o neuronales se verán severamente distorsionados.


# 5. Distribución de Escalada (Target) y Dinámica Temporal

El target representa el nivel del conflicto. Analizaremos su distribución general por país y luego, para solucionar el problema de "puntos constantes en líneas" a lo largo del tiempo, **remuestrearemos los datos** para ver el volumen de días críticos a lo largo de los meses/semanas.


In [ ]:
# Distribución de Escalada por País
col_pais = 'pais' if 'pais' in df.columns else ('country' if 'country' in df.columns else None)

if col_pais and 'target' in df.columns:
    plt.figure(figsize=(10, 6))
    # Porcentaje de clase por país
    crosstab_pct = pd.crosstab(df[col_pais], df['target'], normalize='index') * 100
    ax = crosstab_pct.plot(kind='bar', stacked=True, color=escalada_colors, figsize=(10,6), edgecolor='black')
    
    plt.title('Perfil de Escalada por Actor (Proporción de Días)')
    plt.ylabel('Porcentaje de días (%)')
    plt.xlabel('País')
    plt.legend(['0 - Baja Escalada', '1 - Media Escalada', '2 - Alta Escalada'], title='Target', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=0)
    plt.show()


In [ ]:
if 'fecha' in df.columns and 'target' in df.columns:
    # 1. Densidad Temporal: Contar cuántos días de "Alta Escalada (2)" hubo por Mes
    df_temp = df.set_index('fecha')
    
    # Remuestreamos mensualmente (M) o semanalmente (W-MON) contando frecuencias del target
    escalada_temporal = df_temp.groupby(pd.Grouper(freq='M'))['target'].value_counts().unstack().fillna(0)
    
    # Graficar en barras apiladas temporales
    ax = escalada_temporal.plot(kind='area', stacked=True, color=escalada_colors, figsize=(14, 6), alpha=0.8)
    plt.title('Densidad de la Intensidad del Conflicto a lo largo del Tiempo (Agrupación Mensual)')
    plt.xlabel('Mes')
    plt.ylabel('Cantidad de Días Totales')
    plt.legend(['Baja', 'Media', 'Alta (Crítico)'], title='Nivel de Escalada')
    plt.show()


# 6. Relaciones entre Variables Físicas/Semánticas y la Escalada

Para entender qué variables distinguen mejor si un día será tranquilo o de altísima tensión, graficamos los boxenplots separados por el `target`.


In [ ]:
vars_bivariadas = [v for v in ['n_eventos_7d', 'total_bajas', 'tono_gdelt', 'score_alta_prom'] if v in df.columns]

if vars_bivariadas and 'target' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, var in enumerate(vars_bivariadas):
        # Aplicamos escala symlog a variables de volumen en la vista
        ax = sns.boxenplot(data=df, x='target', y=var, ax=axes[i], palette=escalada_colors)
        if df[var].skew() > 3:
            ax.set_yscale('symlog')
            ax.set_ylabel(f'{var} (SymLog Scale)')
        axes[i].set_title(f'{var} vs Nivel de Escalada')
        
    plt.tight_layout()
    plt.show()


# 7. Selección de Variables para Machine Learning

Evaluamos qué variables se correlacionan numéricamente más fuerte con la escalada. Esto actúa como un filtro inicial de Feature Selection. Luego graficamos la multicolinealidad.


In [ ]:
if 'target' in df.columns:
    num_cols = df.select_dtypes(include=[np.number]).columns
    corr_with_target = df[num_cols].corr()['target'].drop('target').sort_values(ascending=False)
    
    plt.figure(figsize=(12, 8))
    # Tomamos el Top 10 Positivas y Top 10 Negativas para legibilidad
    top_corr = pd.concat([corr_with_target.head(10), corr_with_target.tail(10)])
    
    sns.barplot(x=top_corr.values, y=top_corr.index, palette='coolwarm')
    plt.title('Correlación (Pseudo Point-Biserial) con el Target (Escalada)')
    plt.xlabel('Coeficiente de Correlación de Pearson')
    plt.axvline(x=0, color='black', linewidth=1)
    plt.show()


### Modelo de Importancia de Variables Rápido (Random Forest)
La correlación de Pearson asume relaciones lineales. En conflictos, las relaciones suelen ser no lineales. Para obtener la importancia real de cada feature, entrenamos un pequeño Random Forest.


In [ ]:
if 'target' in df.columns and len(df) > 100:
    # Excluir id, fecha y variables de texto si las hay
    features_to_drop = ['target', 'fecha', 'pais', 'country']
    X_rf = df.drop(columns=[col for col in features_to_drop if col in df.columns])
    X_rf = X_rf.select_dtypes(include=[np.number]).fillna(0)
    y_rf = df['target']
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=5)
    rf.fit(X_rf, y_rf)
    
    importances = pd.Series(rf.feature_importances_, index=X_rf.columns).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='viridis')
    plt.title('Top 15 Features más Importantes (Según Random Forest Proxy)')
    plt.xlabel('Importancia Relativa')
    plt.show()


---
# 8. PIPELINE DE PREPROCESAMIENTO DE DATOS (DATA PROCESSING)

Con todos los descubrimientos del EDA, estamos listos para transformar la data cruda en un dataset perfecto para modelado predictivo (XGBoost, LSTMs, Regresión Logística, etc.).

**Operaciones del Pipeline:**
1. **Extracción de Fechas:** Los modelos (salvo algoritmos específicos de series de tiempo) no entienden el objeto datetime. Extraeremos `mes` y `dia_semana`.
2. **Transformación Logarítmica:** Aplicaremos `np.log1p` a variables altamente asimétricas (`n_eventos`, `total_bajas`, etc.) detectadas en la sección 4, para "normalizar" sus distribuciones.
3. **One-Hot Encoding:** Convertiremos la variable categórica `pais` en columnas binarias (ej. `pais_Iran=1`).
4. **Escalamiento Robusto (RobustScaler):** Dado el gran número de outliers legítimos, `RobustScaler` (que utiliza la mediana y el IQR) es muy superior a StandardScaler o MinMaxScaler en este dominio, ya que no se deja arrastrar por valores extremos.
5. **Generación del Dataset Final.**


In [ ]:
df_processed = df.copy()

# 1. Extracción de Features Temporales
if 'fecha' in df_processed.columns:
    print("-> Extrayendo variables de tiempo...")
    df_processed['mes'] = df_processed['fecha'].dt.month
    df_processed['dia_semana'] = df_processed['fecha'].dt.dayofweek
    df_processed = df_processed.drop(columns=['fecha']) # Se descarta la fecha original

# 2. Transformación Logarítmica de Outliers (Skewed Features)
skewed_candidates = ['n_eventos', 'n_batallas', 'n_explosiones', 'n_protestas', 
                     'total_bajas', 'pob_afectada_1km', 'n_articulos_2026', 
                     'total_bajas_acum5d', 'n_eventos_3d']
vars_to_log = [c for c in skewed_candidates if c in df_processed.columns]

print(f"-> Aplicando transformación Logarítmica (log1p) a {len(vars_to_log)} variables con outliers...")
for var in vars_to_log:
    # np.log1p es log(1 + x), maneja los ceros perfectamente
    df_processed[f'{var}_log'] = np.log1p(df_processed[var])
    # Opcional: Eliminar la original si lo prefieres, aquí la mantenemos o reemplazamos
    df_processed = df_processed.drop(columns=[var])

# 3. Encoding de Variables Categóricas
if col_pais:
    print("-> Aplicando One-Hot Encoding a la variable país...")
    df_processed = pd.get_dummies(df_processed, columns=[col_pais], drop_first=False)

# 4. Manejo de Nulos Residuales (si existieran)
# Se rellenan con 0 o la mediana las variables numéricas
df_processed = df_processed.fillna(df_processed.median(numeric_only=True))

# 5. Escalamiento (RobustScaler)
print("-> Aplicando RobustScaler para igualar magnitudes sin ser afectado por Cisnes Negros...")
target_col = 'target'
features_to_scale = [c for c in df_processed.columns if c != target_col]

scaler = RobustScaler()
df_processed[features_to_scale] = scaler.fit_transform(df_processed[features_to_scale])

# Mostrar el resultado del procesamiento
print("\nProcesamiento finalizado con éxito. Dataset resultante:")
display(df_processed.head())


# 9. Exportación y Conclusiones

El dataset resultante `dataset_procesado_ml.csv` se ha guardado en tu directorio de trabajo. Todas las variables ahora comparten magnitudes comparables y las asimetrías severas fueron suavizadas, garantizando un entrenamiento de modelos estable y rápido.

### Siguientes pasos recomendados para Modelado:
1. **Validación Cruzada Temporal:** Utiliza `TimeSeriesSplit` de `sklearn` en lugar del habitual `KFold` al validar tus modelos, ya que tus datos tienen un orden cronológico implícito.
2. **Manejo del Desbalance (Opcional):** A pesar del preprocesamiento, la clase `2 - Alta Escalada` es minoritaria. Si usas algoritmos de Random Forest o XGBoost, evalúa habilitar el hiperparámetro `class_weight='balanced'`.
3. **Métricas de Éxito:** Utiliza el **Macro F1-Score** o **AUC-PR** (Precision-Recall) como métricas principales, no Accuracy, para asegurarte de que el modelo está aprendiendo a detectar los días de 'Alta Escalada' correctamente.


In [ ]:
# Guardar el dataset listo para ML
output_filename = 'dataset_procesado_ml.csv'
df_processed.to_csv(output_filename, index=False)
print(f"Dataset exportado exitosamente como: {output_filename}")
